In [1]:
# Import required libraries
import os
import pandas as pd

In [2]:
# Load the dataset
df = pd.read_csv('../data/challenge_transcriptomics.tsv', sep='\t')
df = df.drop(columns=['transcriptomics_id'])
df.head()

,participant_id,timepoint,ensembl_gene_id,raw_count,tpm_count,material
0,2024_UGA.ID_128,0,ENSG00000000003,6,0.287,Unknown
1,2024_UGA.ID_128,0,ENSG00000000005,0,0.000,Unknown
2,2024_UGA.ID_128,0,ENSG00000000419,442,10.325,Unknown
3,2024_UGA.ID_128,0,ENSG00000000457,245,7.713,Unknown
4,2024_UGA.ID_128,0,ENSG00000000460,102,3.702,Unknown


In [3]:
# raw_count has actual values; material is uniformly 'Unknown'
# Drop raw_count to keep tpm_count only, consistent with training pipeline
# Drop material (constant 'Unknown')
print('raw_count null fraction:', df['raw_count'].isna().mean())
print('material unique values:', df['material'].unique())
print('timepoints:', sorted(df['timepoint'].unique()))
print('participants:', df['participant_id'].nunique())

raw_count null fraction: 0.0
material unique values: ['Unknown']
timepoints: [np.int64(0), np.int64(7)]
participants: 40


In [4]:
df = df.drop(columns=['raw_count', 'material'])
df.head()

,participant_id,timepoint,ensembl_gene_id,tpm_count
0,2024_UGA.ID_128,0,ENSG00000000003,0.287
1,2024_UGA.ID_128,0,ENSG00000000005,0.000
2,2024_UGA.ID_128,0,ENSG00000000419,10.325
3,2024_UGA.ID_128,0,ENSG00000000457,7.713
4,2024_UGA.ID_128,0,ENSG00000000460,3.702


In [5]:
# Pivot: each row is a (participant_id, timepoint), each gene becomes a column
df_pivot = df.pivot_table(
    index=['participant_id', 'timepoint'],
    columns='ensembl_gene_id',
    values='tpm_count'
)
df_pivot.columns.name = None
df_pivot = df_pivot.reset_index()
print(df_pivot.shape)
df_pivot.head()

(77, 54904)


,participant_id,timepoint,ENSG00000000003,ENSG00000000005,ENSG00000000419,ENSG00000000457,ENSG00000000460,ENSG00000000938,ENSG00000000971,ENSG00000001036,...,ENSG00000310527,ENSG00000310529,ENSG00000310530,ENSG00000310532,ENSG00000310533,ENSG00000310534,ENSG00000310535,ENSG00000310536,ENSG00000310537,ENSG00000310539
0,2024_UGA.ID_077,0,0.251,0.0,11.095,11.281,3.900,510.111,1.709,39.488,...,4.706,0.233,0.000,0.000,0.184,0.0,0.0,0.0,0.000,0.215
1,2024_UGA.ID_077,7,0.293,0.0,18.918,5.259,3.269,494.600,1.041,46.805,...,5.509,0.000,0.000,0.169,0.123,0.0,0.0,0.0,0.000,0.172
2,2024_UGA.ID_086,0,0.034,0.0,9.484,8.148,5.471,638.127,0.504,51.705,...,2.067,0.000,0.817,0.000,0.100,0.0,0.0,0.0,0.031,0.000
3,2024_UGA.ID_086,7,0.835,0.0,11.085,7.744,3.736,780.345,0.903,64.436,...,1.616,0.000,0.000,0.000,0.368,0.0,0.0,0.0,0.113,0.000
4,2024_UGA.ID_128,0,0.287,0.0,10.325,7.713,3.702,385.048,0.369,34.936,...,3.591,0.000,0.000,0.000,0.169,0.0,0.0,0.0,0.172,0.066


In [6]:
# Save the cleaned dataset to the cleaned_data folder
os.makedirs('../cleaned_data', exist_ok=True)
df_pivot.to_csv('../cleaned_data/challenge_transcriptomics_cleaned.csv', index=False)